# Desafio 5 — Pipeline de CPIs (Bronze + Ouro integrado)

## Motivo da unificação
Devido à limitação da API (ausência de endpoint específico para CPIs),
os dados foram extraídos diretamente da tabela `prata.eventos`, já disponível.
Não houve coleta de dados brutos novos, apenas consulta e transformação.

## Estrutura
As análises de ouro foram criadas diretamente a partir da `prata.eventos`,
sem necessidade de camada bronze adicional.

# Exploração do Endpoint de CPIs (Comissões Parlamentares de Inquérito)
Testar endpoints disponíveis para CPIs na API da Câmara.

In [0]:
import requests
import json

url = "https://dadosabertos.camara.leg.br/api/v2/orgaos"
params = {
    "itens": 5,
    "ordem": "ASC",
    "ordenarPor": "sigla"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Total de orgaos: {len(dados['dados'])}")
    print(json.dumps(dados, indent=2, ensure_ascii=False)[:3000])
    for link in dados.get('links', []):
        print(f"  {link['rel']}: {link['href']}")
else:
    print(f"Erro: {response.text}")

In [0]:
import requests
import json

# Testar busca de CPIs pelo nome
url = "https://dadosabertos.camara.leg.br/api/v2/orgaos"
params = {
    "itens": 5,
    "nome": "CPI",
    "ordem": "ASC",
    "ordenarPor": "sigla"
}

response = requests.get(url, params=params)
print(f"Busca por 'CPI': Status {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Resultados: {len(dados['dados'])}")
    if dados['dados']:
        for org in dados['dados'][:3]:
            print(f"  {org['sigla']} - {org['nome']} (ID: {org['id']})")
        for link in dados.get('links', []):
            print(f"  {link['rel']}: {link['href']}")
else:
    print(f"Erro: {response.text}")

# Entrega 1: Tabela Gold de CPIs com Timeline de Eventos

## Origem dos dados
Extraído da tabela `prata.eventos`, filtrando órgãos cujo nome contém "CPI".
Foram encontradas 4 CPIs, todas do ano de 2023.

## Estrutura
Cada linha representa um evento de uma CPI, com data, tipo, duração e período.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.cpis_timeline
USING DELTA
COMMENT 'Camada Ouro - Timeline de eventos das CPIs'
AS
SELECT 
    sigla_orgao AS sigla_cpi,
    nome_orgao AS nome_cpi,
    id_evento,
    data_hora_inicio,
    data_hora_fim,
    DATEDIFF(data_hora_fim, data_hora_inicio) * 24 * 60 AS duracao_minutos,
    descricao_tipo,
    descricao,
    situacao,
    local_camara_nome,
    local_externo,
    url_registro,
    ano,
    mes,
    chave_semana
FROM prata.eventos
WHERE LOWER(nome_orgao) LIKE '%comissão parlamentar de inquérito%'
ORDER BY sigla_orgao, data_hora_inicio

In [0]:
%sql
-- Resumo das CPIs encontradas
SELECT 
    sigla_cpi,
    COUNT(*) AS qtd_eventos,
    MIN(data_hora_inicio) AS inicio,
    MAX(data_hora_inicio) AS fim,
    DATEDIFF(MAX(data_hora_inicio), MIN(data_hora_inicio)) AS duracao_dias,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento
FROM ouro.cpis_timeline
GROUP BY sigla_cpi
ORDER BY qtd_eventos DESC

# Entrega 3: Análise de Duração das CPIs
Compara a duração real de cada CPI com o prazo regimental de 120 dias.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.cpis_duracao
USING DELTA
COMMENT 'Camada Ouro - Analise de duracao das CPIs'
AS
SELECT 
    sigla_cpi,
    nome_cpi,
    MIN(data_hora_inicio) AS data_inicio,
    MAX(data_hora_inicio) AS data_fim,
    DATEDIFF(MAX(data_hora_inicio), MIN(data_hora_inicio)) AS duracao_dias,
    120 AS prazo_regimental_dias,
    CASE 
        WHEN DATEDIFF(MAX(data_hora_inicio), MIN(data_hora_inicio)) > 120 
        THEN 'Excedeu prazo regimental'
        ELSE 'Dentro do prazo'
    END AS status_prazo,
    DATEDIFF(MAX(data_hora_inicio), MIN(data_hora_inicio)) - 120 AS dias_excedentes
FROM ouro.cpis_timeline
GROUP BY sigla_cpi, nome_cpi
ORDER BY duracao_dias DESC

In [0]:
%sql
SELECT 
    sigla_cpi,
    duracao_dias,
    prazo_regimental_dias,
    status_prazo,
    dias_excedentes
FROM ouro.cpis_duracao

# Entrega 5: Comparativo de Produtividade das CPIs
Compara CPIs por quantidade de eventos, tipos de evento e se produziram resultado.
Todas as 4 CPIs foram encerradas, indicando que concluíram os trabalhos.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.cpis_produtividade
USING DELTA
COMMENT 'Camada Ouro - Comparativo de produtividade das CPIs'
AS
SELECT 
    sigla_cpi,
    nome_cpi,
    COUNT(*) AS qtd_eventos,
    COUNT(DISTINCT descricao_tipo) AS qtd_tipos_evento,
    SUM(CASE WHEN descricao_tipo LIKE '%Tomada de Depoimento%' THEN 1 ELSE 0 END) AS qtd_depoimentos,
    SUM(CASE WHEN descricao_tipo LIKE '%Audiência%' THEN 1 ELSE 0 END) AS qtd_audiencias,
    SUM(CASE WHEN descricao_tipo LIKE '%Reunião Deliberativa%' THEN 1 ELSE 0 END) AS qtd_reunioes,
    ROUND(AVG(duracao_minutos), 0) AS duracao_media_min,
    'Encerrada (concluiu os trabalhos)' AS desfecho
FROM ouro.cpis_timeline
GROUP BY sigla_cpi, nome_cpi
ORDER BY qtd_eventos DESC

In [0]:
%sql
SELECT 
    sigla_cpi,
    qtd_eventos,
    qtd_depoimentos,
    qtd_audiencias,
    qtd_reunioes,
    duracao_media_min,
    desfecho
FROM ouro.cpis_produtividade

In [0]:
import requests
import json

url = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
params = {
    "itens": 5,
    "dataInicio": "2023-05-01",
    "dataFim": "2023-07-31",
    "ordem": "ASC",
    "ordenarPor": "id"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Total: {len(dados['dados'])}")
    print(json.dumps(dados['dados'][:2], indent=2, ensure_ascii=False)[:2500])
    for link in dados.get('links', []):
        print(f"  {link['rel']}: {link['href']}")
else:
    print(f"Erro: {response.text}")